In [1]:
from datasets import load_dataset

raw_dataset = load_dataset("tomaarsen/conll2003")

print(raw_dataset)
print(raw_dataset["train"][0])
print(raw_dataset["train"].features['ner_tags'])

README.md:   0%|          | 0.00/15.0k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.24MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  316kB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})
{'id': '0', 'document_id': 1, 'sentence_id': 0, 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}
List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']))


In [7]:
label_names = raw_dataset["train"].features['ner_tags'].feature.names
for i in (0, 4):
  example = raw_dataset["train"][i]
  for word, label in zip(example["tokens"], example["ner_tags"]):
    print(f"{word:12}\t{label_names[label]}")
  print()

EU          	B-ORG
rejects     	O
German      	B-MISC
call        	O
to          	O
boycott     	O
British     	B-MISC
lamb        	O
.           	O

Germany     	B-LOC
's          	O
representative	O
to          	O
the         	O
European    	B-ORG
Union       	I-ORG
's          	O
veterinary  	O
committee   	O
Werner      	B-PER
Zwingmann   	I-PER
said        	O
on          	O
Wednesday   	O
consumers   	O
should      	O
buy         	O
sheepmeat   	O
from        	O
countries   	O
other       	O
than        	O
Britain     	B-LOC
until       	O
the         	O
scientific  	O
advice      	O
was         	O
clearer     	O
.           	O



In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')
example = raw_dataset["train"][0]
inputs = tokenizer(example['tokens'], is_split_into_words=True)

print("Amount of words:", len(example['tokens']))
print("Amount of labels:", len(example['ner_tags']))
print("Amount of tokens:", len(inputs['input_ids']))

for token, word_id in zip(inputs.tokens(), inputs.word_ids()):
  word = '-' if word_id is None else example['tokens'][word_id]
  print(f"{token:12}\t word № {str(word_id):4} {word}")


Amount of words: 9
Amount of labels: 9
Amount of tokens: 12
[CLS]       	 word № None -
EU          	 word № 0    EU
rejects     	 word № 1    rejects
German      	 word № 2    German
call        	 word № 3    call
to          	 word № 4    to
boycott     	 word № 5    boycott
British     	 word № 6    British
la          	 word № 7    lamb
##mb        	 word № 7    lamb
.           	 word № 8    .
[SEP]       	 word № None -


In [9]:
def align_labels_with_tokens(labels, word_ids):
  aligned = []
  previous_word_id = None
  for word_id in word_ids:
    if word_id is None:
      aligned.append(-100)
    elif word_id != previous_word_id:
      previous_word_id = word_id
      aligned.append(labels[word_id])
    else:
      label = labels[word_id]
      if label % 2 == 1:
        label += 1
      aligned.append(label)
  return aligned

def align_first_subtoken_only(labels, word_ids):
  aligned = []
  previous_word_id = None
  for word_id in word_ids:
    if word_id is None:
      aligned.append(-100)
    elif word_id != previous_word_id:
      previous_word_id = word_id
      aligned.append(labels[word_id])
    else:
      aligned.append(-100)
  return aligned

In [10]:
labels = example['ner_tags']
word_ids = inputs.word_ids()
print(labels)
print(align_labels_with_tokens(labels, word_ids))
print(align_first_subtoken_only(labels, word_ids))

[3, 0, 7, 0, 0, 0, 7, 0, 0]
[-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, 0, -100]
[-100, 3, 0, 7, 0, 0, 0, 7, 0, -100, 0, -100]


In [11]:
def tokenize_and_align_labels(examples):
  tokenized = tokenizer(examples['tokens'], truncation=True, is_split_into_words=True)
  tokenized['labels'] = [align_first_subtoken_only(labels, tokenized.word_ids(i)) for i, labels in enumerate(examples['ner_tags'])]
  return tokenized

tokenized_dataset = raw_dataset.map(tokenize_and_align_labels, batched=True, remove_columns=raw_dataset['train'].column_names)

sample = tokenized_dataset['train'][0]
print(tokenized_dataset)
print(len(sample['input_ids']), len(sample['labels']))

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})
12 12


In [12]:
labels = [3, 0]
word_ids = [None, 0, 0, 1, None]

print(align_labels_with_tokens(labels, word_ids))
print(align_first_subtoken_only(labels, word_ids))

[-100, 3, 4, 0, -100]
[-100, 3, -100, 0, -100]


In [13]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

batch = data_collator([tokenized_dataset['train'][i] for i in range(2)])

print("Input shape:", batch['input_ids'].shape)
print("Label shape:", batch['labels'].shape)
print(batch['labels'][1])
print(batch['attention_mask'][1])

Input shape: torch.Size([2, 12])
Label shape: torch.Size([2, 12])
tensor([-100,    1,    2, -100, -100, -100, -100, -100, -100, -100, -100, -100])
tensor([1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0])


In [14]:
%pip install --use-pep517 seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16251 sha256=64c9e923fd83a2914239a6000f00bbb2905c3c9cd5e5ac067d0b0ce6a76db8b2
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval


In [15]:
%pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.3 MB/s eta 0:00:00


In [16]:
import evaluate

metric = evaluate.load('seqeval')
true_labels = [label_names[tag] for tag in example['ner_tags']]
predictions = true_labels.copy()
predictions[2] = 'O'

print(true_labels)
print(predictions)
print(metric.compute(predictions=[predictions], references=[true_labels]))

['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']
['B-ORG', 'O', 'O', 'O', 'O', 'O', 'B-MISC', 'O', 'O']
{'MISC': {'precision': np.float64(1.0), 'recall': np.float64(0.5), 'f1': np.float64(0.6666666666666666), 'number': np.int64(2)}, 'ORG': {'precision': np.float64(1.0), 'recall': np.float64(1.0), 'f1': np.float64(1.0), 'number': np.int64(1)}, 'overall_precision': np.float64(1.0), 'overall_recall': np.float64(0.6666666666666666), 'overall_f1': np.float64(0.8), 'overall_accuracy': 0.8888888888888888}


In [24]:
import numpy as np

def compute_metrics(eval_preds):
  logits, labels = eval_preds
  predicted_ids = np.argmax(logits, axis=-1)

  true_labels = [[label_names[tag] for tag in sentence if tag != -100] for sentence in labels]
  true_predictions = [[label_names[pred] for pred, tag in zip(preds, sentence) if tag != -100]
                      for preds, sentence in zip(predicted_ids, labels)]

  scores = metric.compute(predictions=true_predictions, references=true_labels)
  return {
      'precision': scores['overall_precision'],
      'recall': scores['overall_recall'],
      'f1': scores['overall_f1'],
      'accuracy': scores['overall_accuracy'],
  }


In [32]:
import torch
from transformers import AutoModelForTokenClassification

id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

model = AutoModelForTokenClassification.from_pretrained(
    'bert-base-cased',
    id2label=id2label,
    label2id=label2id,
)

model.eval()
with torch.no_grad():
  outputs = model(**batch)

print(model.config.num_labels)
print(outputs.logits.shape)
print(outputs.loss.item())


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

9
torch.Size([2, 12, 9])
2.1814818382263184


In [33]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
  print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [34]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='bert-finetuned_ner',
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=False,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.052593,0.042363,0.928607,0.936890,0.932730,0.988863
2,0.021997,0.042608,0.947774,0.952878,0.950319,0.991219
3,0.012898,0.041984,0.946691,0.953383,0.950025,0.991375


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5268, training_loss=0.04336822983344215, metrics={'train_runtime': 576.2233, 'train_samples_per_second': 73.102, 'train_steps_per_second': 9.142, 'total_flos': 920771584279074.0, 'train_loss': 0.04336822983344215, 'epoch': 3.0})

In [35]:
test_metrics = trainer.evaluate(
    eval_dataset=tokenized_dataset['test'],
    metric_key_prefix='test',
)
print(test_metrics)

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.012898,0.137924,3,0.907236,0.921211,0.914170,0.982685


{'test_loss': 0.1379241794347763, 'test_precision': 0.9072362685265911, 'test_recall': 0.9212110481586402, 'test_f1': 0.9141702538873758, 'test_accuracy': 0.9826854743189405}


In [36]:
print('validation:', len(tokenized_dataset['validation']))
print('test:', len(tokenized_dataset['test']))

test_output = trainer.predict(tokenized_dataset['test'])
print(len(test_output.predictions))
print(test_output.metrics)

validation: 3250
test: 3453


3453
{'test_loss': 0.1379241794347763, 'test_precision': 0.9072362685265911, 'test_recall': 0.9212110481586402, 'test_f1': 0.9141702538873758, 'test_accuracy': 0.9826854743189405, 'test_runtime': 9.8363, 'test_samples_per_second': 351.047, 'test_steps_per_second': 43.919}


In [37]:
from transformers import pipeline

ner = pipeline(
    'token-classification',
    model=trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy='simple',
    device=0
)

text = "Alice Johnson joined Microsoft in London."
print(ner(text))

for entity in ner(text):
  span = text[entity["start"]: entity["end"]]
  print(f"{entity['entity_group']}: {repr(span)} {round(entity['score'], 3)}")

[{'entity_group': 'PER', 'score': np.float32(0.9993937), 'word': 'Alice Johnson', 'start': 0, 'end': 13}, {'entity_group': 'ORG', 'score': np.float32(0.99912304), 'word': 'Microsoft', 'start': 21, 'end': 30}, {'entity_group': 'LOC', 'score': np.float32(0.9995053), 'word': 'London', 'start': 34, 'end': 40}]
PER: 'Alice Johnson' 0.9990000128746033
ORG: 'Microsoft' 0.9990000128746033
LOC: 'London' 1.0


In [41]:
trainer.save_model("/content/bert-ner-conll2003")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [42]:
from huggingface_hub import login, HfApi

login()
api = HfApi()
repo_id = f"{api.whoami()['name']}/bert-ner-conll2003"

api.create_repo(repo_id=repo_id, private=True, exist_ok=True)
api.upload_folder(
    repo_id=repo_id,
    folder_path="/content/bert-ner-conll2003",
    commit_message="Upload BERT fine-tuned for CoNLL-2003 NER",
)

print(f"https://huggingface.co/{repo_id}")

https://huggingface.co/nerdniwe/bert-ner-conll2003


In [44]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoModelForTokenClassification, get_scheduler
from accelerate import Accelerator

train_loader = DataLoader(tokenized_dataset['train'], batch_size=8, shuffle=True, collate_fn=data_collator)
val_loader = DataLoader(tokenized_dataset['validation'], batch_size=8, shuffle=False, collate_fn=data_collator)

fresh_model = AutoModelForTokenClassification.from_pretrained(
    'bert-base-cased',
    id2label=id2label,
    label2id=label2id,
)

optimizer = AdamW(fresh_model.parameters(), lr=2e-5, weight_decay=0.01)

accelerator = Accelerator()
fresh_model, optimizer, train_loader, val_loader = accelerator.prepare(
    fresh_model, optimizer, train_loader, val_loader
)

num_epochs = 3
num_training_steps = num_epochs * len(train_loader)

scheduler = get_scheduler(
    name='linear',
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

print('Device:', accelerator.device)
print('Number of training steps:', num_training_steps)
print('Batches for epochs:', len(train_loader))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

Device: cuda
Number of training steps: 5268
Batches for epochs: 1756


In [46]:
from tqdm.auto import tqdm

save_dir = "/content/bert-ner-manual"

for epoch in range(num_epochs):
  fresh_model.train()
  train_loss_sum = torch.zeros((), device=accelerator.device)
  for batch in tqdm(train_loader, desc=f'Epoch: {epoch + 1}/{num_epochs}'):
    optimizer.zero_grad()
    outputs = fresh_model(**batch)
    loss = outputs.loss
    accelerator.backward(loss)
    optimizer.step()
    scheduler.step()
    train_loss_sum += loss.detach()

  fresh_model.eval()
  for batch in val_loader:
    with torch.no_grad():
      outputs = fresh_model(**batch)
    predictions = outputs.logits.argmax(dim=-1)
    labels = batch['labels']
    predictions = accelerator.pad_across_processes(predictions, dim=1, pad_index=-100)
    labels = accelerator.pad_across_processes(labels, dim=1, pad_index=-100)
    predictions, labels = accelerator.gather_for_metrics((predictions, labels))
    predictions = predictions.cpu().numpy()
    labels = labels.cpu().numpy()
    true_labels = [
        [label_names[tag] for tag in sentence if tag != -100]
        for sentence in labels
    ]
    true_predictions = [
        [label_names[pred] for pred, tag in zip(preds, sentence) if tag != -100]
        for preds, sentence in zip(predictions, labels)
    ]
    metric.add_batch(predictions=true_predictions, references=true_labels)

  scores = metric.compute()
  accelerator.print(
      f"Epoch {epoch+1}: "
      f"train_loss: {train_loss_sum.item() / len(train_loader):.4f} "
      f"precision: {scores['overall_precision']:.4f} "
      f"recall: {scores['overall_recall']:.4f} "
      f"f1: {scores['overall_f1']:.4f} "
  )
  accelerator.wait_for_everyone()
  unwrapped_model = accelerator.unwrap_model(fresh_model)
  unwrapped_model.save_pretrained(
      save_dir,
      is_main_process=accelerator.is_main_process,
      save_function=accelerator.save)
  if accelerator.is_main_process:
    tokenizer.save_pretrained(save_dir)

Epoch: 1/3:   0%|          | 0/1756 [00:00<?, ?it/s]

Epoch 1: train_loss: 0.0257 precision: 0.9336 recall: 0.9492 f1: 0.9413 


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch: 2/3:   0%|          | 0/1756 [00:00<?, ?it/s]

Epoch 2: train_loss: 0.0130 precision: 0.9403 recall: 0.9519 f1: 0.9461 


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch: 3/3:   0%|          | 0/1756 [00:00<?, ?it/s]

Epoch 3: train_loss: 0.0096 precision: 0.9403 recall: 0.9519 f1: 0.9461 


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [47]:
test_loader = DataLoader(tokenized_dataset['test'], batch_size=8, shuffle=False, collate_fn=data_collator)
test_loader = accelerator.prepare(test_loader)

fresh_model.eval()
for batch in test_loader:
  with torch.no_grad():
    logits = fresh_model(**batch).logits
  predicted_ids = logits.argmax(dim=-1)
  labels = batch['labels']
  predicted_ids = accelerator.pad_across_processes(predicted_ids, dim=1, pad_index=-100)
  labels = accelerator.pad_across_processes(labels, dim=1, pad_index=-100)
  predicted_ids, labels = accelerator.gather_for_metrics((predicted_ids, labels))
  predicted_ids = predicted_ids.cpu().numpy()
  labels = labels.cpu().numpy()

  true_labels = [
      [label_names[tag] for tag in sentence if tag != -100]
      for sentence in labels
  ]
  true_predictions = [
      [label_names[pred] for pred, tag in zip(preds, sentence) if tag != -100]
      for preds, sentence in zip(predicted_ids, labels)]
  metric.add_batch(predictions=true_predictions, references=true_labels)
test_scores = metric.compute()
print({
    name: test_scores[f'overall_{name}']
    for name in ['precision', 'recall', 'f1', 'accuracy']
})

{'precision': np.float64(0.9025703369225425), 'recall': np.float64(0.9201487252124646), 'f1': np.float64(0.9112747676661406), 'accuracy': 0.9823624421233983}


In [48]:
manual_repo_id = f"{api.whoami()['name']}/bert-ner-coll2003-manual"
api.create_repo(repo_id=manual_repo_id, private=True, exist_ok=True)
api.upload_folder(
    repo_id=manual_repo_id,
    folder_path=save_dir,
    commit_message="Upload BERT fine-tuned for CoNLL-2003 NER",
)
print(f"https://huggingface.co/{manual_repo_id}")

https://huggingface.co/nerdniwe/bert-ner-coll2003-manual


In [49]:
from transformers import pipeline

manual_ner = pipeline(
    'token-classification',
    model=manual_repo_id,
    token=True,
    aggregation_strategy='simple',
    device=0
)

text = "Alice Johnson joined Microsoft in London."
print(manual_ner(text))

config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  431MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/352 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/669k [00:00<?, ?B/s]

[{'entity_group': 'PER', 'score': np.float32(0.9991282), 'word': 'Alice Johnson', 'start': 0, 'end': 13}, {'entity_group': 'ORG', 'score': np.float32(0.9990283), 'word': 'Microsoft', 'start': 21, 'end': 30}, {'entity_group': 'LOC', 'score': np.float32(0.99840325), 'word': 'London', 'start': 34, 'end': 40}]
